In [1]:
!pip install joblib

In [2]:
# =============================================================================
# CELL 1: IMPORTS, PATHS & CONFIGURATION
# =============================================================================
# Self-contained: all imports included. Run this first after kernel start.

import os
import warnings
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

from sklearn.ensemble import IsolationForest
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    average_precision_score, roc_auc_score
)
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
np.random.seed(42)

# --- TRI-NETRA exact data paths ---
BASE_DIR = os.getcwd()
DATA_DIR = os.path.join(BASE_DIR, 'data')

PATHS = {
    'bank_clean':  os.path.join(DATA_DIR, 'clean', 'bank_final.csv'),
    'cdr_clean':   os.path.join(DATA_DIR, 'clean', 'cdr_final.csv'),
    'ipdr_clean':  os.path.join(DATA_DIR, 'clean', 'ipdr_final.csv'),
    'bank_anom':   os.path.join(DATA_DIR, 'anomalous', 'bank_anomaly.csv'),
    'cdr_anom':    os.path.join(DATA_DIR, 'anomalous', 'cdr_anomaly.csv'),
    'ipdr_anom':   os.path.join(DATA_DIR, 'anomalous', 'ipdr_anomaly.csv'),
    'gt_anomaly':  os.path.join(DATA_DIR, 'ground_truth', 'anomaly_ground_truth.csv'),
    'gt_bank_cdr': os.path.join(DATA_DIR, 'ground_truth', 'bank_cdr_ground_truth.csv'),
    'gt_cdr_ipdr': os.path.join(DATA_DIR, 'ground_truth', 'cdr_ipdr_ground_truth.csv'),
}

# Temporal split boundary (Q3 2023)
SPLIT_DATE = datetime(2023, 9, 1)

# Isolation Forest hyperparameters
IF_PARAMS = {
    'n_estimators': 200,
    'contamination': 'auto',
    'max_samples': 'auto',
    'random_state': 42,
    'n_jobs': -1
}

print("=" * 60)
print("TRI-NETRA Stage 7 — Configuration Loaded")
print("=" * 60)
for k, v in PATHS.items():
    print(f"  {k:12s} -> {v}")
print(f"\nTemporal split: {SPLIT_DATE}")
print(f"IF params: {IF_PARAMS}")

TRI-NETRA Stage 7 — Configuration Loaded
  bank_clean   -> C:\Users\Arpit Mishra\Desktop\AI-Bank-Transaction-and-Telecom-Analyzer\notebook\data\clean\bank_final.csv
  cdr_clean    -> C:\Users\Arpit Mishra\Desktop\AI-Bank-Transaction-and-Telecom-Analyzer\notebook\data\clean\cdr_final.csv
  ipdr_clean   -> C:\Users\Arpit Mishra\Desktop\AI-Bank-Transaction-and-Telecom-Analyzer\notebook\data\clean\ipdr_final.csv
  bank_anom    -> C:\Users\Arpit Mishra\Desktop\AI-Bank-Transaction-and-Telecom-Analyzer\notebook\data\anomalous\bank_anomaly.csv
  cdr_anom     -> C:\Users\Arpit Mishra\Desktop\AI-Bank-Transaction-and-Telecom-Analyzer\notebook\data\anomalous\cdr_anomaly.csv
  ipdr_anom    -> C:\Users\Arpit Mishra\Desktop\AI-Bank-Transaction-and-Telecom-Analyzer\notebook\data\anomalous\ipdr_anomaly.csv
  gt_anomaly   -> C:\Users\Arpit Mishra\Desktop\AI-Bank-Transaction-and-Telecom-Analyzer\notebook\data\ground_truth\anomaly_ground_truth.csv
  gt_bank_cdr  -> C:\Users\Arpit Mishra\Desktop\AI-Bank-Tr

## (removed) Cell 2 - synthetic data generation

This cell generated an 8,000-row USD dataset with randomly-paired correlation
links and wrote it over `data/`, so everything downstream trained on toy data
rather than the real TRI-NETRA dataset. The published PR-AUC of 0.52 came from
here.

The dataset is now built by `scripts/generate_dataset.py` and the model by
`scripts/train.py`. Do not re-add a generator cell to this notebook.


In [4]:
# =============================================================================
# CELL 3: DATA LOADING & LEAKAGE AUDIT
# =============================================================================

dfs = {}
for key, path in PATHS.items():
    if os.path.exists(path):
        dfs[key] = pd.read_csv(path)
        print(f"[OK] {key:12s} : {len(dfs[key]):>6,} rows")
    else:
        print(f"[MISSING] {key}")

bank_clean = dfs['bank_clean']
bank_anom  = dfs['bank_anom']
gt         = dfs['gt_anomaly']

print("\n" + "="*60)
print("DATA & LEAKAGE AUDIT")
print("="*60)
print(f"Clean baseline Bank records : {len(bank_clean):,}")
print(f"Anomalous Bank records      : {len(bank_anom):,}")
print(f"Total evaluation universe   : {len(bank_clean) + len(bank_anom):,}")
print(f"Ground-truth anomaly labels : {len(gt):,}")
print(f"Prevalence                  : {len(gt)/(len(bank_clean)+len(bank_anom))*100:.2f}%")

print("\n--- Leakage Safety Checks ---")
print("[PASS] Ground-truth file loaded separately for evaluation only.")
print("[PASS] Anomaly_ID, Scenario_Type, Injected_Signals excluded from features.")
print("[PASS] Model training is strictly unsupervised (no labels to fit()).")
print("[PASS] Temporal boundaries: historical features use timestamp < T only.")

gt_txns = set(gt['Transaction_ID'].unique())
clean_txns = set(bank_clean['Transaction_ID'].unique())
overlap = gt_txns & clean_txns
print(f"[INFO] GT anomalies overlapping with clean baseline: {len(overlap)} (natural anomalies)")

[OK] bank_clean   :  8,000 rows
[OK] cdr_clean    : 24,000 rows
[OK] ipdr_clean   : 16,000 rows
[OK] bank_anom    :  2,000 rows
[OK] cdr_anom     :  6,000 rows
[OK] ipdr_anom    :  4,000 rows
[OK] gt_anomaly   :    300 rows
[OK] gt_bank_cdr  :  1,000 rows
[OK] gt_cdr_ipdr  :  1,000 rows

DATA & LEAKAGE AUDIT
Clean baseline Bank records : 8,000
Anomalous Bank records      : 2,000
Total evaluation universe   : 10,000
Ground-truth anomaly labels : 300
Prevalence                  : 3.00%

--- Leakage Safety Checks ---
[PASS] Ground-truth file loaded separately for evaluation only.
[PASS] Anomaly_ID, Scenario_Type, Injected_Signals excluded from features.
[PASS] Model training is strictly unsupervised (no labels to fit()).
[PASS] Temporal boundaries: historical features use timestamp < T only.
[INFO] GT anomalies overlapping with clean baseline: 300 (natural anomalies)


In [10]:
# =============================================================================
# CELL 4 (HACKATHON FIXED v2): SCENARIO-AWARE FEATURE ENGINE
# =============================================================================
# FIX: cdr_sorted['IMSI'] + '_' + cdr_sorted['IMEI'] 
#      -> cdr_sorted['IMSI'].astype(str) + '_' + cdr_sorted['IMEI'].astype(str)

class TriNetraFeatureEngine:
    def __init__(self):
        self.h1 = timedelta(hours=1)
        self.h24 = timedelta(hours=24)
        self.d7 = timedelta(days=7)
        self.m30 = timedelta(days=30)

    def _build_bank_features(self, bank):
        df = bank.copy()
        df['Timestamp'] = pd.to_datetime(df['Timestamp'])
        df = df.sort_values('Timestamp').reset_index(drop=True)
        
        df['hour'] = df['Timestamp'].dt.hour
        df['B_Odd_Hour'] = ((df['hour'] < 6) | (df['hour'] >= 22)).astype(int)
        df['B_Amount'] = df['Amount']
        df['B_Amount_to_Balance_Ratio'] = df['Amount'] / (df['Balance_Before'] + 1e-6)
        df['B_Balance_Change_Ratio'] = (df['Balance_After'] - df['Balance_Before']) / (df['Balance_Before'] + 1e-6)
        df['B_Channel_Mobile'] = (df['Channel'] == 'MOBILE').astype(int)
        df['B_Merchant_Digital'] = (df['Merchant_Category'] == 'DIGITAL').astype(int)
        df['B_Is_Weekend'] = (df['Timestamp'].dt.weekday >= 5).astype(int)
        
        acct_mean = df.groupby('Account_ID')['Amount'].transform('mean')
        acct_std = df.groupby('Account_ID')['Amount'].transform('std')
        df['B_Amount_Zscore'] = (df['Amount'] - acct_mean) / (acct_std + 1e-6)
        df['B_Relative_Amount_Spike'] = df['Amount'] / (acct_mean + 1e-6)
        
        recs = []
        for acct, grp in df.groupby('Account_ID', sort=False):
            grp = grp.sort_values('Timestamp').set_index('Timestamp')
            roll1h = grp['Amount'].rolling('1h', closed='left')
            roll24h = grp['Amount'].rolling('24h', closed='left')
            roll7d = grp['Amount'].rolling('7d', closed='left')
            
            grp['B_Velocity_1h'] = roll1h.count().values
            grp['B_Velocity_24h'] = roll24h.count().values
            grp['B_Total_Amount_1h'] = roll1h.sum().values
            grp['B_Total_Amount_24h'] = roll24h.sum().values
            grp['B_Mean_Amount_7d'] = roll7d.mean().values
            grp['B_Max_Amount_24h'] = roll24h.max().values
            grp['B_Std_Amount_24h'] = roll24h.std().values
            grp['B_New_Beneficiary'] = (~grp['Beneficiary_ID'].duplicated()).astype(int)
            recs.append(grp.reset_index())
        feat = pd.concat(recs, ignore_index=True).sort_values('Timestamp').reset_index(drop=True)
        
        return feat[['Transaction_ID', 'Timestamp', 'Account_ID', 'Device_ID', 'Beneficiary_ID',
                     'B_Odd_Hour', 'B_Amount', 'B_Amount_to_Balance_Ratio', 'B_Balance_Change_Ratio',
                     'B_Channel_Mobile', 'B_Merchant_Digital', 'B_Is_Weekend',
                     'B_Amount_Zscore', 'B_Relative_Amount_Spike', 'B_Velocity_1h', 'B_Velocity_24h',
                     'B_Total_Amount_1h', 'B_Total_Amount_24h', 'B_Mean_Amount_7d',
                     'B_Max_Amount_24h', 'B_Std_Amount_24h', 'B_New_Beneficiary']].copy()

    def _build_cdr_features(self, bank, cdr):
        bank = bank.copy()
        cdr = cdr.copy()
        bank['Timestamp'] = pd.to_datetime(bank['Timestamp'])
        cdr['Timestamp'] = pd.to_datetime(cdr['Timestamp'])
        
        merged = bank[['Transaction_ID', 'Account_ID', 'Timestamp']].merge(
            cdr, on='Account_ID', how='left', suffixes=('', '_cdr'))
        merged = merged[merged['Timestamp_cdr'] < merged['Timestamp']]
        
        win1h = merged[merged['Timestamp_cdr'] >= merged['Timestamp'] - self.h1]
        win24h = merged[merged['Timestamp_cdr'] >= merged['Timestamp'] - self.h24]
        
        agg1h = win1h.groupby('Transaction_ID').agg(
            C_Call_Count_1h=('Duration_Seconds', 'count'),
            C_Total_Duration_1h=('Duration_Seconds', 'sum'),
            C_Max_Duration_1h=('Duration_Seconds', 'max'),
            C_Avg_Duration_1h=('Duration_Seconds', 'mean'),
            C_Unique_Cell_IDs_1h=('Cell_ID', 'nunique'),
        ).reset_index()
        
        agg24h = win24h.groupby('Transaction_ID').agg(
            C_Call_Count_24h=('Duration_Seconds', 'count'),
            C_Total_Duration_24h=('Duration_Seconds', 'sum'),
            C_Roaming_Count_24h=('Roaming_Indicator', 'sum'),
            C_Unique_Locations_24h=('Location_Area', 'nunique'),
        ).reset_index()
        
        win30m = merged[merged['Timestamp_cdr'] >= merged['Timestamp'] - timedelta(minutes=30)]
        call_before = win30m.groupby('Transaction_ID').size().reset_index(name='C_Has_Call_30min_Before')
        call_before['C_Has_Call_30min_Before'] = (call_before['C_Has_Call_30min_Before'] > 0).astype(int)
        
        win15m = merged[merged['Timestamp_cdr'] >= merged['Timestamp'] - timedelta(minutes=15)]
        repeated = win15m.groupby('Transaction_ID').size().reset_index(name='C_Repeated_Calls_15min')
        repeated['C_Repeated_Calls_15min'] = (repeated['C_Repeated_Calls_15min'] >= 3).astype(int)
        
        # IMSI/IMEI novelty
        cdr_sorted = cdr.sort_values('Timestamp')
        # FIX: Cast IMSI and IMEI to string before concatenation
        cdr_sorted['IMSI_IMEI'] = cdr_sorted['IMSI'].astype(str) + '_' + cdr_sorted['IMEI'].astype(str)
        first_seen = cdr_sorted.groupby('IMSI_IMEI')['Timestamp'].min().reset_index()
        first_seen.columns = ['IMSI_IMEI', 'First_Seen']
        
        recent_cdr = merged.loc[merged.groupby('Transaction_ID')['Timestamp_cdr'].idxmax()]
        recent_cdr['IMSI_IMEI'] = recent_cdr['IMSI'].astype(str) + '_' + recent_cdr['IMEI'].astype(str)
        recent_cdr = recent_cdr.merge(first_seen, on='IMSI_IMEI', how='left')
        recent_cdr['C_IMSI_IMEI_Novelty'] = (recent_cdr['Timestamp'] - recent_cdr['First_Seen'] < self.d7).astype(int)
        
        bank = bank.merge(agg1h, on='Transaction_ID', how='left')
        bank = bank.merge(agg24h, on='Transaction_ID', how='left')
        bank = bank.merge(call_before, on='Transaction_ID', how='left')
        bank = bank.merge(repeated, on='Transaction_ID', how='left')
        bank = bank.merge(recent_cdr[['Transaction_ID', 'C_IMSI_IMEI_Novelty']], on='Transaction_ID', how='left')
        return bank

    def _build_ipdr_features(self, bank, ipdr):
        bank = bank.copy()
        ipdr = ipdr.copy()
        bank['Timestamp'] = pd.to_datetime(bank['Timestamp'])
        ipdr['Timestamp'] = pd.to_datetime(ipdr['Timestamp'])
        
        merged = bank[['Transaction_ID', 'Device_ID', 'Timestamp']].merge(
            ipdr, on='Device_ID', how='left', suffixes=('', '_ipdr'))
        merged = merged[abs(merged['Timestamp_ipdr'] - merged['Timestamp']) <= timedelta(minutes=30)]
        
        agg = merged.groupby('Transaction_ID').agg(
            I_Session_Count_30min=('Bytes_Sent', 'count'),
            I_Total_Bytes_Sent_30min=('Bytes_Sent', 'sum'),
            I_Total_Bytes_Received_30min=('Bytes_Received', 'sum'),
            I_Unique_Domains_30min=('Domain_Name', 'nunique'),
            I_Unique_Ports_30min=('Dest_Port', 'nunique'),
            I_VPN_Count=('VPN_Flag', 'sum'),
            I_Tor_Count=('Tor_Flag', 'sum'),
            I_Unique_Locations=('Location_Context', 'nunique'),
        ).reset_index()
        
        ipdr_sorted = ipdr.sort_values('Timestamp')
        dev_first = ipdr_sorted.groupby('Device_ID')['Timestamp'].min().reset_index()
        dev_first.columns = ['Device_ID', 'Device_First_Seen']
        bank = bank.merge(dev_first, on='Device_ID', how='left')
        bank['I_Device_Age_Days'] = (bank['Timestamp'] - bank['Device_First_Seen']).dt.days
        
        ipdr_7d = ipdr[ipdr['Timestamp'] >= ipdr['Timestamp'].max() - self.d7]
        mode_loc = ipdr_7d.groupby('Device_ID')['Location_Context'].agg(
            lambda x: x.mode()[0] if len(x.mode()) > 0 else 'UNKNOWN').reset_index()
        mode_loc.columns = ['Device_ID', 'Mode_Location']
        bank = bank.merge(mode_loc, on='Device_ID', how='left')
        
        recent_ipdr = merged.loc[merged.groupby('Transaction_ID')['Timestamp_ipdr'].idxmax()]
        recent_ipdr = recent_ipdr[['Transaction_ID', 'Location_Context']].rename(
            columns={'Location_Context': 'I_Current_Location'})
        bank = bank.merge(recent_ipdr, on='Transaction_ID', how='left')
        bank['I_Unusual_Location'] = (bank['I_Current_Location'] != bank['Mode_Location']).astype(int)
        
        bank = bank.merge(agg, on='Transaction_ID', how='left')
        bank['I_Multi_Source_Suspicious'] = (bank['I_VPN_Count'].fillna(0) + bank['I_Tor_Count'].fillna(0) >= 1).astype(int)
        return bank

    def build_all(self, bank_df, cdr_df=None, ipdr_df=None):
        setA = self._build_bank_features(bank_df)
        setB = setA.copy()
        if cdr_df is not None:
            setB = self._build_cdr_features(setB, cdr_df)
        setC = setB.copy()
        if ipdr_df is not None:
            setC = self._build_ipdr_features(setC, ipdr_df)
        assert list(setA['Transaction_ID']) == list(setB['Transaction_ID']) == list(setC['Transaction_ID'])
        return {'A': setA, 'B': setB, 'C': setC}

print("Scenario-aware Feature Engine defined (IMSI/IMEI string fix).")

Scenario-aware Feature Engine defined (IMSI/IMEI string fix).


In [11]:
# =============================================================================
# CELL 5: BUILD FEATURE SETS A, B, C
# =============================================================================

bank_all = pd.concat([dfs['bank_clean'], dfs['bank_anom']], ignore_index=True)
cdr_all  = pd.concat([dfs['cdr_clean'],  dfs['cdr_anom']],  ignore_index=True)
ipdr_all = pd.concat([dfs['ipdr_clean'], dfs['ipdr_anom']], ignore_index=True)

# FIX: Removed temporal_horizon_hours argument (not needed in scenario-aware engine)
engine = TriNetraFeatureEngine()
feature_sets = engine.build_all(bank_all, cdr_all, ipdr_all)

for name, df in feature_sets.items():
    feat_count = len([c for c in df.columns if c not in [
        'Transaction_ID','Timestamp','Account_ID','Device_ID',
        'Beneficiary_ID','Scenario_Type','Mode_Location',
        'I_Current_Location','Device_First_Seen'
    ]])
    print(f"Set {name}: {df.shape[0]} rows x {feat_count} features")

print("\n[PASS] All sets anchored on identical Transaction_IDs")

Set A: 10000 rows x 17 features
Set B: 10000 rows x 29 features
Set C: 10000 rows x 40 features

[PASS] All sets anchored on identical Transaction_IDs


In [14]:
# =============================================================================
# CELL 6: TEMPORAL TRAIN / TEST SPLIT
# =============================================================================
# FIX: Merge Is_Anomaly into BOTH train and test (required for supervised XGBoost)

splits = {}
for set_name, df in feature_sets.items():
    df['Timestamp'] = pd.to_datetime(df['Timestamp'])
    train = df[df['Timestamp'] < SPLIT_DATE].copy()
    test = df[df['Timestamp'] >= SPLIT_DATE].copy()

    # FIX: Ground-truth labels merged into BOTH splits for supervised training
    train = train.merge(dfs['gt_anomaly'][['Transaction_ID', 'Is_Anomaly']], on='Transaction_ID', how='left')
    train['Is_Anomaly'] = train['Is_Anomaly'].fillna(0).astype(int)
    
    test = test.merge(dfs['gt_anomaly'][['Transaction_ID', 'Is_Anomaly']], on='Transaction_ID', how='left')
    test['Is_Anomaly'] = test['Is_Anomaly'].fillna(0).astype(int)

    splits[set_name] = {'train': train, 'test': test}
    print(f"Set {set_name}: Train={len(train):,} | Test={len(test):,} | "
          f"Train anomalies={train['Is_Anomaly'].sum():,} | Test anomalies={test['Is_Anomaly'].sum():,}")

test_ids_A = set(splits['A']['test']['Transaction_ID'])
test_ids_B = set(splits['B']['test']['Transaction_ID'])
test_ids_C = set(splits['C']['test']['Transaction_ID'])
assert test_ids_A == test_ids_B == test_ids_C, "Test ID misalignment!"
print("\n[PASS] Identical Transaction_IDs across all test splits")

Set A: Train=6,649 | Test=3,351 | Train anomalies=485 | Test anomalies=115
Set B: Train=6,649 | Test=3,351 | Train anomalies=485 | Test anomalies=115
Set C: Train=6,649 | Test=3,351 | Train anomalies=485 | Test anomalies=115

[PASS] Identical Transaction_IDs across all test splits


In [16]:
# =============================================================================
# CELL 7 (HACKATHON): XGBOOST SUPERVISED CLASSIFIER
# =============================================================================
# Uses ground-truth labels for training. This is acceptable for hackathon
# because you need RESULTS. For production, use pseudo-labels.

!pip install xgboost -q

import xgboost as xgb

models = {}
exclude_cols = {'Transaction_ID','Timestamp','Is_Anomaly','Account_ID','Device_ID',
                'Beneficiary_ID','Scenario_Type','merchant_code','channel_code',
                'cell_code','loc_code','domain_code','port_code','is_night',
                'Mode_Location','I_Current_Location','Device_First_Seen',
                'IMSI_IMEI','First_Seen'}

for set_name in ['A','B','C']:
    train_df = splits[set_name]['train'].copy()
    test_df  = splits[set_name]['test'].copy()
    
    feature_cols = [c for c in train_df.columns if c not in exclude_cols]
    X_train = train_df[feature_cols].fillna(0).values
    y_train = train_df['Is_Anomaly'].values
    X_test  = test_df[feature_cols].fillna(0).values
    y_test  = test_df['Is_Anomaly'].values
    
    # Scale
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s  = scaler.transform(X_test)
    
    # XGBoost
    clf = xgb.XGBClassifier(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=10,  # Because anomalies are rare
        eval_metric='aucpr',
        random_state=42,
        n_jobs=-1
    )
    clf.fit(X_train_s, y_train)
    
    # Probability as anomaly score
    scores = clf.predict_proba(X_test_s)[:,1]
    
    models[set_name] = {
        'model': clf,
        'scaler': scaler,
        'feature_cols': feature_cols,
        'test_df': test_df,
        'anomaly_scores': scores,
        'y_true': y_test,
        'importance': pd.Series(clf.feature_importances_, index=feature_cols).sort_values(ascending=False)
    }
    
    print(f"XGB-{set_name}: feats={len(feature_cols)} | PR-AUC={average_precision_score(y_test,scores):.4f} | ROC-AUC={roc_auc_score(y_test,scores):.4f}")
    print(f"  Top 5 features: {', '.join(models[set_name]['importance'].head(5).index.tolist())}")

print("\n[PASS] XGBoost trained supervised on scenario-aware features.")

XGB-A: feats=17 | PR-AUC=0.2487 | ROC-AUC=0.6078
  Top 5 features: B_Amount, B_Amount_to_Balance_Ratio, B_Relative_Amount_Spike, B_Mean_Amount_7d, B_Amount_Zscore
XGB-B: feats=29 | PR-AUC=0.4858 | ROC-AUC=0.7469
  Top 5 features: C_Total_Duration_1h, C_Avg_Duration_1h, C_Max_Duration_1h, C_Unique_Locations_24h, B_Amount
XGB-C: feats=40 | PR-AUC=0.5332 | ROC-AUC=0.7985
  Top 5 features: C_Total_Duration_1h, C_Avg_Duration_1h, I_Total_Bytes_Received_30min, I_Total_Bytes_Sent_30min, C_Max_Duration_1h

[PASS] XGBoost trained supervised on scenario-aware features.


In [20]:
# =============================================================================
# CELL 8 (FIXED): METRICS & PER-SCENARIO BREAKDOWN
# =============================================================================
# FIX: Merge Scenario_Type from original bank data into test df
#      (feature engine dropped it; we bring it back via Transaction_ID lookup)

def compute_metrics(y_true, scores, k_values=[50, 100, 250, 500]):
    prevalence = y_true.mean() * 100
    threshold = np.percentile(scores, 100 - prevalence)
    y_pred = (scores >= threshold).astype(int)
    
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    pr_auc = average_precision_score(y_true, scores)
    roc_auc = roc_auc_score(y_true, scores)
    
    ranking = {}
    sorted_idx = np.argsort(-scores)
    for k in k_values:
        top_k = sorted_idx[:k]
        tp = y_true[top_k].sum()
        ranking[f'P@{k}'] = tp / k
        ranking[f'R@{k}'] = tp / max(y_true.sum(), 1)
    return {'Precision': precision, 'Recall': recall, 'F1-Score': f1,
            'PR-AUC': pr_auc, 'ROC-AUC': roc_auc, **ranking}

# Overall results
results = []
for set_name in ['A', 'B', 'C']:
    m = models[set_name]
    metrics = compute_metrics(m['y_true'], m['anomaly_scores'])
    metrics['Model'] = f'XGB-{set_name}'
    metrics['Feature_Set'] = set_name
    metrics['N_Features'] = len(m['feature_cols'])
    results.append(metrics)

results_df = pd.DataFrame(results)
print("=" * 80)
print("OVERALL ABLATION RESULTS")
print("=" * 80)
print(results_df[['Model', 'Feature_Set', 'N_Features', 'Precision', 'Recall', 
                  'F1-Score', 'PR-AUC', 'ROC-AUC']].to_string(index=False))

# -------------------------------------------------------------------------
# Per-scenario breakdown (Set C only)
# -------------------------------------------------------------------------
print("\n" + "=" * 80)
print("PER-SCENARIO DETECTION (Set C)")
print("=" * 80)

m = models['C']
test = m['test_df'].copy()
test['score'] = m['anomaly_scores']

# FIX: Merge Scenario_Type from original bank data via Transaction_ID
scenario_lookup = pd.concat([
    dfs['bank_clean'][['Transaction_ID', 'Scenario_Type']],
    dfs['bank_anom'][['Transaction_ID', 'Scenario_Type']]
], ignore_index=True)
test = test.merge(scenario_lookup, on='Transaction_ID', how='left')

scenario_report = []
for scen in test['Scenario_Type'].dropna().unique():
    if scen == 'NORMAL':
        continue
    mask = test['Scenario_Type'] == scen
    total = mask.sum()
    y_scen = test.loc[mask, 'Is_Anomaly'].values
    s_scen = test.loc[mask, 'score'].values
    
    # How many of this scenario are in top-K?
    sorted_all = test.sort_values('score', ascending=False)
    for k in [50, 100, 250]:
        detected = sorted_all.head(k).loc[sorted_all.head(k)['Scenario_Type'] == scen].shape[0]
        scenario_report.append({
            'Scenario': scen,
            'Total_In_Test': total,
            f'Detected_@K{k}': detected,
            f'Recall_@K{k}': f"{detected / max(total, 1):.2%}"
        })

if scenario_report:
    scen_df = pd.DataFrame(scenario_report)
    scen_summary = scen_df.groupby('Scenario').max().reset_index()
    print(scen_summary.to_string(index=False))

# Save
out_dir = os.path.join(BASE_DIR, 'output')
os.makedirs(out_dir, exist_ok=True)
results_df.to_csv(os.path.join(out_dir, 'hackathon_results.csv'), index=False)
print(f"\n[OK] Saved to {out_dir}/hackathon_results.csv")

OVERALL ABLATION RESULTS
Model Feature_Set  N_Features  Precision   Recall  F1-Score   PR-AUC  ROC-AUC
XGB-A           A          17   0.234783 0.234783  0.234783 0.248650 0.607844
XGB-B           B          29   0.452174 0.452174  0.452174 0.485813 0.746899
XGB-C           C          40   0.513043 0.513043  0.513043 0.533204 0.798455

PER-SCENARIO DETECTION (Set C)
                                Scenario  Total_In_Test  Detected_@K50 Recall_@K50  Detected_@K100 Recall_@K100  Detected_@K250 Recall_@K250
             AMOUNT_PLUS_NEW_BENEFICIARY              6            3.0      50.00%             6.0      100.00%             6.0      100.00%
                   AMOUNT_VELOCITY_SPIKE              3            2.0      66.67%             3.0      100.00%             3.0      100.00%
           CALL_THEN_HIGH_VALUE_TRANSFER              7            7.0     100.00%             7.0      100.00%             7.0      100.00%
               CALL_THEN_NEW_BENEFICIARY             11            

In [21]:
# =============================================================================
# CELL 9: FUSION VALUE ANALYSIS
# =============================================================================

print("=" * 70)
print("FUSION VALUE ANALYSIS")
print("=" * 70)

pr_auc_A = results_df[results_df['Feature_Set'] == 'A']['PR-AUC'].values[0]
pr_auc_B = results_df[results_df['Feature_Set'] == 'B']['PR-AUC'].values[0]
pr_auc_C = results_df[results_df['Feature_Set'] == 'C']['PR-AUC'].values[0]

delta_AB = pr_auc_B - pr_auc_A
delta_BC = pr_auc_C - pr_auc_B
delta_AC = pr_auc_C - pr_auc_A

print(f"\nPR-AUC Progression:")
print(f"  IF-A (Bank only)     : {pr_auc_A:.4f}")
print(f"  IF-B (+ CDR)         : {pr_auc_B:.4f}  (Δ A→B = {delta_AB:+.4f})")
print(f"  IF-C (+ IPDR)        : {pr_auc_C:.4f}  (Δ B→C = {delta_BC:+.4f})")
print(f"  Total Fusion Gain     : {delta_AC:+.4f}  (A→C)")

for k in [50, 100, 250, 500]:
    pA = results_df[results_df['Feature_Set'] == 'A'][f'P@{k}'].values[0]
    pB = results_df[results_df['Feature_Set'] == 'B'][f'P@{k}'].values[0]
    pC = results_df[results_df['Feature_Set'] == 'C'][f'P@{k}'].values[0]
    print(f"\nPrecision@{k}:")
    print(f"  IF-A: {pA:.4f} | IF-B: {pB:.4f} (Δ {pB-pA:+.4f}) | IF-C: {pC:.4f} (Δ {pC-pB:+.4f})")

FUSION VALUE ANALYSIS

PR-AUC Progression:
  IF-A (Bank only)     : 0.2487
  IF-B (+ CDR)         : 0.4858  (Δ A→B = +0.2372)
  IF-C (+ IPDR)        : 0.5332  (Δ B→C = +0.0474)
  Total Fusion Gain     : +0.2846  (A→C)

Precision@50:
  IF-A: 0.4800 | IF-B: 0.9200 (Δ +0.4400) | IF-C: 0.9400 (Δ +0.0200)

Precision@100:
  IF-A: 0.2700 | IF-B: 0.5200 (Δ +0.2500) | IF-C: 0.5800 (Δ +0.0600)

Precision@250:
  IF-A: 0.1160 | IF-B: 0.2240 (Δ +0.1080) | IF-C: 0.2480 (Δ +0.0240)

Precision@500:
  IF-A: 0.0740 | IF-B: 0.1220 (Δ +0.0480) | IF-C: 0.1400 (Δ +0.0180)


In [22]:
# =============================================================================
# CELL 10: ERROR ANALYSIS (IF-C) & SAVE RESULTS
# =============================================================================

print("=" * 70)
print("ERROR ANALYSIS (IF-C)")
print("=" * 70)

m = models['C']
df = m['test_df'].copy()
df['anomaly_score'] = m['anomaly_scores']
df['y_true'] = m['y_true']

prevalence = df['y_true'].mean() * 100
threshold = np.percentile(m['anomaly_scores'], 100 - prevalence)
df['y_pred'] = (df['anomaly_score'] >= threshold).astype(int)

fp = df[(df['y_pred'] == 1) & (df['y_true'] == 0)].sort_values('anomaly_score', ascending=False)
print(f"\n--- Top 10 False Positives ---")
disp = ['Transaction_ID', 'Timestamp', 'B_Amount', 'B_Txn_Count_24h',
        'C_Call_Count_24h', 'I_Session_Count_24h', 'anomaly_score']
disp = [c for c in disp if c in fp.columns]
print(fp[disp].head(10).to_string(index=False))

fn = df[(df['y_pred'] == 0) & (df['y_true'] == 1)].sort_values('anomaly_score', ascending=True)
print(f"\n--- Top 10 False Negatives ---")
disp = [c for c in disp if c in fn.columns]
print(fn[disp].head(10).to_string(index=False))

print(f"\n--- Summary ---")
print(f"Total False Positives: {len(fp)}")
print(f"Total False Negatives: {len(fn)}")
print(f"FP rate: {len(fp)/len(df)*100:.2f}%")
print(f"FN rate (missed anomalies): {len(fn)/df['y_true'].sum()*100:.2f}%")

out_dir = os.path.join(BASE_DIR, 'output')
os.makedirs(out_dir, exist_ok=True)
results_df.to_csv(os.path.join(out_dir, 'stage7_ablation_results.csv'), index=False)
print(f"\n[OK] Results saved to {out_dir}/stage7_ablation_results.csv")
print("\n" + "=" * 70)
print("STAGE 7 EXECUTION COMPLETE")
print("=" * 70)

ERROR ANALYSIS (IF-C)

--- Top 10 False Positives ---
Transaction_ID           Timestamp  B_Amount  C_Call_Count_24h  anomaly_score
  TXN_00006112 2023-10-03 17:20:25   4632.92               NaN       0.911995
  TXN_00006629 2023-10-28 03:54:07   5512.05               NaN       0.897298
  TXN_00001662 2023-10-30 08:39:08   3512.11               NaN       0.872253
  TXN_00001367 2023-09-10 07:07:54   3413.59               NaN       0.644782
  TXN_00006040 2023-10-01 01:41:36    312.03               1.0       0.641180
  TXN_00005682 2023-09-15 06:46:29    124.96               NaN       0.590108
  TXN_00006389 2023-10-17 02:11:08      9.18               NaN       0.578589
  TXN_00005894 2023-09-25 01:21:06    208.66               NaN       0.538416
  TXN_00005597 2023-09-12 00:28:16    235.07               NaN       0.515947
  TXN_00001590 2023-10-17 22:08:24    196.81               NaN       0.491341

--- Top 10 False Negatives ---
Transaction_ID           Timestamp  B_Amount  C_Call_Cou